In [1]:
# Import Libraries
import rasterio
import numpy as np
import os
import json
from datetime import datetime
from scipy.ndimage import binary_opening, binary_closing
print("Libraries imported successfully")

Libraries imported successfully


In [2]:
CONFIG = {
    "past_ndwi": "../data/processed/past_ndwi.tif",
    "recent_ndwi": "../data/processed/recent_ndwi.tif",
    "output_folder": "../data/processed",
    "ndwi_threshold": 0.3,
    "pixel_resolution": 30
}

In [3]:
def load_ndwi(path):
    with rasterio.open(path) as src:
        ndwi = src.read(1)
        profile = src.profile
    return ndwi, profile

past_ndwi, profile = load_ndwi(CONFIG["past_ndwi"])
recent_ndwi, _ = load_ndwi(CONFIG["recent_ndwi"])

print("NDWI rasters loaded successfully")

RasterioIOError: ../data/processed/past_ndwi.tif: No such file or directory

In [ ]:
past_water = past_ndwi > CONFIG["ndwi_threshold"]
recent_water = recent_ndwi > CONFIG["ndwi_threshold"]

print("Water masks created")

Water masks created


In [ ]:
structure = np.ones((3, 3))

past_water_clean = binary_opening(past_water, structure)
past_water_clean = binary_closing(past_water_clean, structure)

recent_water_clean = binary_opening(recent_water, structure)
recent_water_clean = binary_closing(recent_water_clean, structure)

print("Noise filtering applied")

Noise filtering applied


In [ ]:
flood_expansion = np.logical_and(
    recent_water_clean,
    np.logical_not(past_water_clean)
)

print("Flood expansion mask generated")

Flood expansion mask generated


In [ ]:
pixel_area_m2 = CONFIG["pixel_resolution"] ** 2
pixel_area_km2 = pixel_area_m2 / 1_000_000

past_area = np.sum(past_water_clean) * pixel_area_km2
recent_area = np.sum(recent_water_clean) * pixel_area_km2
flood_area = np.sum(flood_expansion) * pixel_area_km2

percent_increase = (
    ((recent_area - past_area) / past_area) * 100
    if past_area != 0 else 0
)

print("Past Area (sq km):", round(past_area, 2))
print("Recent Area (sq km):", round(recent_area, 2))
print("Flood Expansion (sq km):", round(flood_area, 2))
print("Increase (%):", round(percent_increase, 2))

Past Area (sq km): 0.01
Recent Area (sq km): 0.19
Flood Expansion (sq km): 0.19
Increase (%): 2200.0


In [ ]:
profile.update(dtype=rasterio.uint8, count=1)

output_path = f"{CONFIG['output_folder']}/flood_expansion.tif"

with rasterio.open(output_path, "w", **profile) as dst:
    dst.write(flood_expansion.astype(rasterio.uint8), 1)

print("Flood expansion raster saved")

Flood expansion raster saved


In [ ]:
metadata = {
    "ndwi_threshold": CONFIG["ndwi_threshold"],
    "pixel_resolution": CONFIG["pixel_resolution"],
    "past_area_km2": round(past_area, 2),
    "recent_area_km2": round(recent_area, 2),
    "flood_expansion_km2": round(flood_area, 2),
    "percent_increase": round(percent_increase, 2),
    "timestamp": datetime.utcnow().isoformat()
}

with open("../output/detection_metadata.json", "w") as f:
    json.dump(metadata, f, indent=4)

print("Detection metadata saved")

Detection metadata saved
